In [1]:
!python --version

Python 3.12.5


# SynTagRus

In [2]:
import os
import pandas as pd
import re
import json

In [ ]:
corpus_location = "./raw/SynTagRus2022/"

In [ ]:
bodyTextDf_loc = './data/bodyTextDf.json'
bodyLibDf_loc = './data/bodyLibDf.json'
infDict_loc = './data/infDict.json'
infDictDf_loc = './data/infDictDf.json'

# Make

In [5]:
feat_dict = {
    "pos" : ["S", "A", "V", "ADV", "NUM", "PR", "COM", "CONJ", "P", "PART", "INTJ", "NID"],
    "animacy" : ["ОД", "НЕОД"],
    "gender" : ["МУЖ", "ЖЕН", "СРЕД"],
    "number" : ["ЕД", "МН"],
    "case" : ["ИМ", "РОД", "ПАРТ", "ДАТ", "ВИН", "ТВОР", "ПР", "МЕСТН"],
    "adjective_level" : ["СРАВ", "ПРЕВ"],
    "shortness" : ["КР"],
    "verb_rep" : ["ИНФ", "ПРИЧ", "ДЕЕПР"],
    "mood" : ["ИЗЪЯВ", "ПОВ"],
    "aspect" : ["НЕСОВ", "СОВ"],
    "person" : ["1-Л", "2-Л", "3-Л"],
    "passive" : ["СТРАД"],
    "word_formation" : ["СЛ"],
    "mod_comparative" : ["СМЯГ"]
    }

In [ ]:
feat_def_dict = {
    "pos" : {
        "S": 2, # noun
        "A": 0, # adjective
        "V": 11, # verb
        "ADV": 1, # adverb
        "NUM": 6, # number
        "PR": 9, # preposition
        "COM": 2, # 
        "CONJ": 3, # conjunction
        "P": 5, # pronoun
        "PART": 8, # particle
        "INTJ": 4, # interjection
        "NID": 12,  # no ID
    },
    "subst_animacy" : {
        "ОД": True,
        "НЕОД": False
        },
    "gram_gender" : {
        "МУЖ": 0,
        "ЖЕН": 1,
        "СРЕД": 2
        },
    "gram_number" : {
        "ЕД": 0,
        "МН": 1
        },
    "subst_case" : {
        "ИМ": 0,
        "РОД": 1,
        "ПАРТ": 8,
        "ДАТ": 3,
        "ВИН": 2,
        "ТВОР": 4,
        "ПР": 5,
        "МЕСТН": 7
        },
    "adjv_comp_type" : ["СРАВ", "ПРЕВ"],
    "adjv_short" : ["КР"],
    "verb_infinitive" : {
        "ИНФ": True
    },
    "verb_mood" : {
        "ИЗЪЯВ": 0,
        "ПОВ": 1
    },
    "verb_aspect" : {
        "НЕСОВ": 0,
        "СОВ": 1
        },
    "verb_conj_person" : {
        "1-Л": 1,
        "2-Л": 2,
        "3-Л": 3
        },
    "other" : ["СТРАД", "СЛ", "ПРИЧ", "ДЕЕПР", "СМЯГ"]
    }

In [6]:
enum_feat_dict = {}
for key, val in feat_dict.items():
    enum_feat = {key: dict(enumerate(val))}
    enum_feat_dict.update(enum_feat)
#enum_feat_dict
rev_enum_feat_dict = {}
for key, val in enum_feat_dict.items():
    val_dict = {}
    for k, v in val.items():
        val_dict.update({v:k})
    rev_enum_feat_dict.update({key:val_dict})
#rev_enum_feat_dict

In [7]:
def list_files_recursive(path='.', ff=None, file_ext=".tgt", re_pattern=False, sort=False):

    # instantiate list for all files
    file_list = []
    
    def filter_files(path):
        for root, dirs, files in os.walk(path):
            # Remove unwanted dirs in-place
            dirs[:] = [d for d in dirs if not d.startswith('.') and d != '.ipynb_checkpoints']
        
            for f in files:
                if f.endswith(file_ext) and not f.startswith("."):
                    full_path = os.path.join(root, f)
                    file_list.append(full_path)
        
    if ff:
        filter_files(path)
    else:
        # for loop through the dirs
        for root, dirs, files in os.walk(path):
            for file in files:
                # check for existence of RE pattern to be applied
                if re_pattern != None:
                    f = re.compile(re_pattern)
                    if f.search(file):
                        file_list.append(os.path.join(root, file))
                else:
                    file_list.append(os.path.join(root, file))

    if sort:
        file_list.sort()
    
    return file_list

In [8]:
def parseW(file_list):
    
    #
    parsedBodyDf = pd.DataFrame(columns=['doc_id'])

    #
    parsedInfDict = {}
    
    #
    idx = 0
    
    #
    print(len(file_list))
    
    # function to assign codes to category columns
    def assign_codes(code_list):
        result = {cat: [] for cat in feat_dict.keys()}
        for code in code_list:
            category = code_to_category.get(code)
            if category:
                result[category].append(code)
        # Join multiple codes (if any) into a space-separated string
        return {cat: ' '.join(codes) if codes else None for cat, codes in result.items()}
    
    for file in file_list:
        # make dictionary from 'inf' tag contents (author, date, source, title)
        parsedInf = pd.read_xml(file, xpath="/text/inf").T[0].to_dict()
        
        #
        wDf = pd.read_xml(file, xpath="/text/body/S/W")
        
        #
        Slines = wDf.loc[wDf.ID == 1].index
        
        #
        wDf.columns = [x.lower() for x in wDf.columns]
        
        #
        wDf = wDf.rename(columns={'id':'word_id'})
        
        #
        wDf.loc[wDf.loc[wDf.word_id == 1].index, 'sent_id'] = range(1,len(Slines)+1)
        
        #
        wDf.sent_id = wDf.sent_id.ffill().astype('int')
        
        #
        wDf.lemma = wDf.lemma.apply(lambda x: x.lower())
    
        # invert the dictionary to map code to category
        code_to_category = {}
        for cat, codes in feat_dict.items():
            for code in codes:
                code_to_category[code] = cat
        
        # apply the function to each row, collect the new columns
        new_cols = wDf['feat'].str.split().apply(assign_codes).apply(pd.Series)
        
        # concatenate the original df with new_cols
        wDf = pd.concat([wDf, new_cols], axis=1)
        
        # drop original 'feat' column
        wDf = wDf.drop(columns=['feat'])
        
        # add idx to df
        wDf.loc[:, 'doc_id'] = idx
        
        # give the index a name: t(oken)_id
        wDf.index.name = 'doc_tok_id'
        
        # reset index for workability
        wDf = wDf.reset_index()

        #
        parsedInfDict[idx] = parsedInf

        #
        parsedBodyDf = pd.concat([parsedBodyDf, wDf], axis=0, ignore_index=True)
        
        #
        idx += 1

    
    parsedBodyLibDf = parsedBodyDf[[
        'doc_id', 'sent_id', 'word_id', 'doc_tok_id', 'dom', 'link', 'lemma',
        'ksname', 'nodetype', 'extracomm', 'status'
    ]]

    parsedBodyTextDf = parsedBodyDf[[
        'doc_id', 'doc_tok_id', 'w', 'pos', 'animacy', 'gender', 'number', 
        'case', 'adjective_level', 'shortness', 'verb_rep', 'mood', 
        'aspect', 'person', 'passive', 'word_formation', 'mod_comparative'
    ]]    
    
    return parsedInfDict, parsedBodyLibDf, parsedBodyTextDf

In [ ]:
def parseW_optimized(file_list, feat_dict):
    """
    Parses a list of XML files into DataFrames in a more optimized way.

    Args:
        file_list (list): A list of file paths to the XML files.
        feat_dict (dict): A dictionary mapping feature categories to their codes.

    Returns:
        tuple: A tuple containing (parsed_inf_dict, parsedBodyLibDf, parsedBodyTextDf).
    """
    # --- 1. Pre-computation (Moved outside the loop) ---
    # Invert the dictionary once before the loop begins.
    code_to_category = {
        code: category for category, codes in feat_dict.items() for code in codes
    }

    # Helper function is also defined once.
    def assign_codes(code_list):
        if not isinstance(code_list, list):
            return {cat: None for cat in feat_dict.keys()}
        
        result = {cat: [] for cat in feat_dict.keys()}
        for code in code_list:
            category = code_to_category.get(code)
            if category:
                result[category].append(code)
        return {cat: ' '.join(codes) if codes else None for cat, codes in result.items()}

    # --- 2. Process files and collect DataFrames in a list ---
    all_body_dfs = []
    parsed_inf_dict = {}

    print(f"Processing {len(file_list)} files...")
    for doc_id, file in enumerate(file_list):
        # Process metadata
        parsed_inf = pd.read_xml(file, xpath="/text/inf").T[0].to_dict()
        parsed_inf_dict[doc_id] = parsed_inf

        # Process main content from files like A_on_myatezhnyi.tgt
        wDf = pd.read_xml(file, xpath="/text/body/S/W")
        
        # --- 3. Vectorized and Chained Operations ---
        wDf.columns = wDf.columns.str.lower()
        wDf = wDf.rename(columns={'id': 'word_id'})
        
        # Calculate sentence IDs more efficiently using cumsum()
        sent_starts = wDf['word_id'] == 1
        wDf['sent_id'] = sent_starts.cumsum()
        
        wDf['lemma'] = wDf['lemma'].str.lower()

        # Process the 'feat' column
        if 'feat' in wDf.columns:
            new_cols_df = wDf['feat'].str.split().apply(assign_codes).apply(pd.Series)
            wDf = pd.concat([wDf, new_cols_df], axis=1)
            wDf = wDf.drop(columns=['feat'])

        wDf['doc_id'] = doc_id
        all_body_dfs.append(wDf)

    # --- 4. Single Concatenation After the Loop ---
    if not all_body_dfs:
        return {}, pd.DataFrame(), pd.DataFrame()
        
    parsedBodyDf = pd.concat(all_body_dfs, ignore_index=True)
    parsedBodyDf = parsedBodyDf.reset_index().rename(columns={'index': 'doc_tok_id'})

    # --- 5. Final DataFrame Slicing ---
    lib_cols = [
        'doc_id', 'sent_id', 'word_id', 'doc_tok_id', 'dom', 'link', 'lemma',
        'ksname', 'nodetype', 'extracomm', 'status'
    ]
    text_cols = [
        'doc_id', 'doc_tok_id', 'w', 'pos', 'animacy', 'gender', 'number', 
        'case', 'adjective_level', 'shortness', 'verb_rep', 'mood', 
        'aspect', 'person', 'passive', 'word_formation', 'mod_comparative'
    ]
    
    # Filter columns to only those that exist to avoid KeyErrors
    parsedBodyLibDf = parsedBodyDf[[col for col in lib_cols if col in parsedBodyDf.columns]]
    parsedBodyTextDf = parsedBodyDf[[col for col in text_cols if col in parsedBodyDf.columns]]
    
    return parsed_inf_dict, parsedBodyLibDf, parsedBodyTextDf

In [10]:
file_list = list_files_recursive(corpus_location, ff=True, sort=True)

In [ ]:
parsedInfDict, parsedBodyLibDf, parsedBodyTextDf = parseW_optimized(file_list, feat_dict)

# Explore

In [ ]:
parsedBodyLibDf = pd.read_json(bodyLibDf_loc)

In [12]:
parsedBodyLibDf

,doc_id,sent_id,word_id,doc_tok_id,dom,link,lemma,ksname,nodetype,extracomm,status
0,0,1,1,0,_root,None,а,А1,None,None,None
1,0,1,2,1,4,предик,он,ОН,None,None,None
2,0,1,3,2,2,оп-опред,мятежный,None,None,None,None
3,0,1,4,3,1,соч-союзн,просить,ПРОСИТЬ2,None,None,None
4,0,1,5,4,4,1-компл,буря,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
1441401,1278,5,9,1441401,6,подч-союзн,быть,БЫТЬ,None,None,None
1441402,1278,5,10,1441402,9,аналит,проходить,ПРОХОДИТЬ4,None,None,None
1441403,1278,5,11,1441403,13,опред,всемирный,None,None,None,None
1441404,1278,5,12,1441404,13,опред,экономический,None,None,None,None


In [11]:
parsedBodyTextDf = pd.read_json(bodyTextDf_loc).T

In [12]:
parsedBodyTextDf.loc[parsedBodyTextDf['pos'] == 'COM']

,doc_id,doc_tok_id,w,pos,animacy,gender,number,case,adjective_level,shortness,verb_rep,mood,aspect,person,passive,word_formation,mod_comparative
6149,2,6149,радио,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
36962,17,36962,мини,COM,None,None,None,None,None,None,None,None,None,None,None,СЛ,None
51199,27,51199,бета,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
51206,27,51206,Бета,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
104827,70,104827,гамма,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429078,1200,1429078,Альфа,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
1429089,1200,1429089,Альфа,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
1429120,1200,1429120,Альфа,COM,None,None,None,None,None,None,None,None,None,None,None,None,None
1429124,1200,1429124,Альфа,COM,None,None,None,None,None,None,None,None,None,None,None,None,None


### sentence_docs

In [ ]:
sentenceDocsDf = pd.DataFrame(parsedInfDict).T
sentenceDocsDf

### sentence_token_meta

In [ ]:
parsedBodyLibDf['doc_tok_id'] = parsedBodyLibDf['doc_tok_id'].astype('Int64')
parsedBodyLibDf['word_id'] = parsedBodyLibDf['word_id'].astype('Int64')
parsedBodyLibDf['sent_id'] = parsedBodyLibDf['sent_id'].astype('Int64')
parsedBodyLibDf

In [ ]:
parsedBodyLibDf[['doc_id', 'sent_id']].drop_duplicates().reset_index().drop('index', axis=1)

### sentence_token_gram

In [ ]:
parsedBodyTextDf['doc_tok_id'] = parsedBodyTextDf['doc_tok_id'].astype('Int64')
parsedBodyTextDf

### sentence_tokens

In [ ]:
#sentenceTokensDf = pd.DataFrame(parsedBodyTextDf.w.unique())
sentenceTokensDf = pd.DataFrame(parsedBodyTextDf.w.str.lower().unique())
sentenceTokensDf

In [ ]:
parsedBodyTextDf.to_json(path_or_buf=bodyTextDf_loc, orient='index')

In [ ]:
parsedBodyLibDf.to_json(path_or_buf=bodyLibDf_loc, orient='index')

In [ ]:
pd.DataFrame(parsedInfDict).T.to_json(path_or_buf=infDictDf_loc, orient='index')

In [ ]:
with open(infDict_loc, "w") as file:
    json.dump(parsedInfDict, file, indent=4)

In [ ]:
# Open and read the JSON file
with open(infDict_loc, 'r') as file:
    parsedInfDict = json.load(file)

In [ ]:
parsedBodyLibDf = pd.read_json(bodyLibDf_loc)

In [ ]:
parsedBodyTextDf = pd.read_json(bodyTextDf_loc)

In [ ]:
parsedBodyLibDf.T

In [ ]:
parsedBodyTextDf = parsedBodyTextDf.T

In [ ]:
parsedBodyTextDf.loc[parsedBodyTextDf.lemma == "человек"].sample(5)